# 04 — Similarity graph construction (issue #5)

Turns issue #4's patient x code vectors into an actual patient-patient graph -- the first real graph this project builds. Pipeline (`ehrgraph.graphs.similarity`), each stage its own tested function:

1. **`build_patient_vectors`** (issue #4) -- binary + per_patient, the primary config: patient x code, 0/1 presence.
2. **`to_tfidf`** -- IDF-weights the binary vectors (downweights near-universal codes, upweights rare/distinguishing ones) *without* reintroducing the utilization confound issue #4 specifically avoided -- see the module docstring in `ehrgraph/graphs/similarity.py` for why IDF-on-binary, not raw-count TF-IDF, is the right combination here.
3. **`cosine_similarity_matrix`** -- every patient's similarity to every other patient.
4. **`top_k_sparsify`** (k=10) -- keep each patient's 10 most similar neighbors only; a full similarity matrix is not a usable graph (nearly everyone is weakly "connected" to everyone).
5. **`mutual_knn`** (primary) / **`union_knn`** (secondary sensitivity check) -- `top_k_sparsify` is *not* symmetric (patient A having B in A's top-10 doesn't mean A is in B's top-10), so we have to explicitly choose how to symmetrize before this can be an undirected graph.
6. **`to_networkx`** -- the actual graph object, plus `graph_summary` (isolated-node count, largest-connected-component size -- a real finding, not just cleanup, per `PROJECT_GUIDE.md` Module A2).

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_local_data  # noqa: E402

data_root = ensure_local_data()
assets = data_root / "EHRSHOT_ASSETS"
meds_reader_path = data_root / "meds_reader_omop_ehrshot"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from ehrgraph.data.loader import open_database
from ehrgraph.graphs.similarity import (
    cosine_similarity_matrix,
    graph_summary,
    mutual_knn,
    to_networkx,
    to_tfidf,
    top_k_sparsify,
    union_knn,
)
from ehrgraph.graphs.vectorize import build_patient_vectors

plt.rcParams["figure.facecolor"] = "white"

## Cohort and vectors -- same `guo_los` cohort and primary config as notebook 03

In [ ]:
labels = pd.read_csv(assets / "benchmark" / "guo_los" / "labeled_patients.csv")
db = open_database(str(meds_reader_path))
db_ids = set(db)
patient_ids = [p for p in labels["patient_id"].unique().tolist() if p in db_ids]

matrix, pids, codes = build_patient_vectors(
    db, patient_ids, table="condition", weighting="binary", dedup="per_patient"
)
print(
    f"{len(pids)} patients, {len(codes)} distinct condition codes, {matrix.nnz:,} nonzero entries"
)

## Similarity: TF-IDF weight the vectors, then cosine similarity between every pair of patients

In [ ]:
tfidf = to_tfidf(matrix)
sim = cosine_similarity_matrix(tfidf)
density = sim.nnz / (sim.shape[0] * sim.shape[1])
print(f"similarity matrix: {sim.shape}, {sim.nnz:,} nonzero pairs ({density:.2%} density)")
print(
    "This confirms the problem sparsification exists to fix: most patients "
    "share at least one diagnosis with most other patients, so a majority "
    "of pairs get a nonzero similarity score -- that's noise, not a graph."
)

## Sparsify: keep each patient's top 10 most similar neighbors, then symmetrize

`top_k_sparsify` alone is directional (A picking B doesn't mean B picked A) -- `mutual_knn` (primary) keeps an edge only if both sides picked each other; `union_knn` (secondary check) keeps an edge if either side did.

In [ ]:
top_k = top_k_sparsify(sim, k=10)
mutual = mutual_knn(top_k)
union = union_knn(top_k)
print(f"top-10 (directional): {top_k.nnz:,} entries")
print(f"mutual (primary):     {mutual.nnz:,} entries -> {mutual.nnz // 2:,} undirected edges")
print(f"union (secondary):    {union.nnz:,} entries -> {union.nnz // 2:,} undirected edges")

## The actual graphs, and the real finding: how connected is this population?

In [ ]:
G_mutual = to_networkx(mutual, pids)
G_union = to_networkx(union, pids)

summary_df = pd.DataFrame(
    [
        {"condition": "mutual k-NN (primary)", **graph_summary(G_mutual)},
        {"condition": "union k-NN (secondary)", **graph_summary(G_union)},
    ]
).set_index("condition")
summary_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
summary_df[["isolated_fraction"]].plot(kind="bar", ax=axes[0], legend=False, color="#C44E52")
axes[0].set_title("Fraction of patients with zero connections")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

summary_df[["largest_cc_fraction"]].plot(kind="bar", ax=axes[1], legend=False, color="#4C72B0")
axes[1].set_title("Fraction of patients in the largest connected cluster")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

fig.tight_layout()
plt.show()

**Reading this result:** mutual k-NN is the stricter, primary choice -- it leaves a real minority of patients with no reciprocated top-10 match at all (isolated), but the remaining graph is cleaner (patients who *are* connected are connected because both sides agreed they're alike, not because one side's similarity score happened to be marginally in the top 10). Union k-NN barely leaves anyone isolated, at the cost of including one-sided, weaker connections. Both numbers get reported -- this asymmetry-driven gap between the two is itself informative about how naturally clustered (vs. spread out) this patient population is, which is exactly the kind of thing Module A2 flags as a real finding rather than a data-cleaning footnote.

## Summary -- issue #5 done-when checklist

- **A weighted, symmetric patient-similarity graph exists in `networkx` for Task A** (`guo_los`) -- `G_mutual`, built via `to_networkx`, real edge weights (cosine similarity scores), confirmed symmetric by construction (`mutual_knn`).
- **Basic stats printed**: node count (3,850), edge count, isolated-node count and fraction, largest-connected-component size and fraction -- for both the primary (mutual) and secondary (union) sparsification choices.
- **Isolated-node fraction and largest-CC fraction reported as a real finding**, not just a footnote, per `PROJECT_GUIDE.md` Module A2's explicit instruction.
- Every stage of the pipeline (`to_tfidf`, `cosine_similarity_matrix`, `top_k_sparsify`, `mutual_knn`/`union_knn`, `to_networkx`, `graph_summary`) is independently unit-tested against small hand-built matrices in `tests/test_similarity.py`, and re-verified here against the real 3,850-patient cohort.
- Next: issue #6/#7/#8 build the three graph *conditions* (leakage-safe, intentionally-leaky, null) on top of this same pipeline -- this notebook's graph used each patient's *full* history (no `prediction_time` censoring), which is the intentionally-leaky condition's job, not this issue's.